## The objective of this notebook is to perform modeling on the miRNA data

In [94]:
from sklearn.svm import SVC 
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression 
from sklearn.decomposition import PCA 
from sklearn.model_selection import train_test_split,StratifiedKFold 
from sklearn.metrics import accuracy_score,confusion_matrix,precision_score,recall_score,f1_score 
from sklearn.preprocessing import StandardScaler,Normalizer
from sklearn.dummy import DummyClassifier
from statistics import mean, stdev 

In [95]:
# Reading cleaned data

data = pd.read_csv("cleaned_data/processed_data_v1.csv")
data_modeling = (
    data
    .rename(columns = {"Healthy":"Disease_State"})
    .sort_values(by = ["Disease_State"])
    .reset_index(drop = True)
)

<IPython.core.display.Javascript object>

F:\Python\lib\site-packages\IPython\core\interactiveshell.py:3457: DtypeWarning: Columns (1) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [96]:
data_modeling.head()

,Disease_State,Cancer,hsa-miR-486-5p,hsa-miR-451a,hsa-miR-92a-3p,hsa-miR-181a-5p,hsa-miR-99a-5p,hsa-miR-21-5p,hsa-miR-26a-5p,hsa-miR-22-3p,...,hsa-miR-3658,hsa-miR-3620-5p,hsv1-miR-H4-5p,hsa-miR-4798-3p,hsa-miR-5702,hsa-miR-548ao-5p,hsa-miR-4790-5p,hsa-miR-4684-5p,hsa-miR-4539,hsa-miR-10525-3p
0,0.0,False,4382.03,20197.87,2742.05,26685.93,189688.17,1508.20,4230.72,90675.11,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,False,916.51,3.24,67163.77,56893.55,9.72,149564.01,80458.33,48540.62,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,False,1131.59,543.49,68046.34,81203.58,39.54,23735.96,96816.65,15205.46,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,False,1357.35,180.81,67620.79,48461.67,113.72,54721.72,107791.57,12507.52,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,False,15873.02,0.00,65759.64,106575.96,0.00,52154.20,106575.96,29478.46,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [97]:
data_modeling["Disease_State"].value_counts()

0.0    1002
1.0     470
Name: Disease_State, dtype: int64

In [98]:
# Extracting the independent and dependent variables

ind_vars=data_modeling.drop(['Cancer','Disease_State'],axis=1).to_numpy()
dep_vars=data_modeling['Disease_State'].to_numpy()

In [99]:
# Defining model objects

baseline = DummyClassifier(strategy="stratified")
model1 = LogisticRegression(C=10.0, solver='lbfgs', max_iter=1000, random_state=42)
model2 = SVC(C=10.0, gamma='scale', random_state=42)
model3 = RandomForestClassifier(n_estimators=150, max_depth=12, criterion='entropy', random_state=42)
model4 = GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, subsample=0.8, random_state=42)

In [100]:
def cancer_modeling(ind_vars, dep_vars, model):

    # Predefining empty lists for metrics
    accuracy_list = []
    precision_list = []
    recall_list = []
    f1_list = []
    
    # Initializing StratifiedKFold object with 10 folds 
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
        
    for train_index, test_index in skf.split(ind_vars, dep_vars):

        X_train, X_test = ind_vars[train_index], ind_vars[test_index]
        y_train, y_test = dep_vars[train_index], dep_vars[test_index]
        
        # Defining a StandardScaler object and standardizing miRNA RPM values
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Dimensionality reduction using PCA to reduce multiple miRNAs into meaningful principle components

        pca = PCA(n_components=0.95)  # Retains 95% of variance 
        X_train_reduced = pca.fit_transform(X_train_scaled)
        X_test_reduced = pca.transform(X_test_scaled)
        
        # Fit the external model object passed into the function
        model.fit(X_train_reduced, y_train)
        
        y_pred = model.predict(X_test_reduced)
        
        # Calculating metrics
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        
        accuracy_list.append(acc)
        precision_list.append(prec)
        recall_list.append(rec)
        f1_list.append(f1)
        
    return mean(accuracy_list), mean(precision_list), mean(recall_list), mean(f1_list), stdev(accuracy_list), stdev(precision_list), stdev(recall_list), stdev(f1_list)

In [106]:
acc,prec,rec,f1,std_acc,std_prec,std_rec,std_f1 = cancer_modeling(ind_vars,dep_vars,baseline)

model_type = "baseline"

print(f"Accuracy of {model_type} model is",(acc*100),"±",(std_acc*100))
print(f"Precision of {model_type} is",(prec*100),"±",(std_prec*100))
print(f"Recall of {model_type} is",(rec*100),"±",(std_rec*100))
print(f"F1-score of {model_type} is",(f1*100),"±",(std_f1*100))

Accuracy of baseline model is 56.31503952932524 ± 2.3892910714825315
Precision of baseline is 55.795741755701755 ± 2.58617624571428
Recall of baseline is 56.31503952932524 ± 2.3892910714825315
F1-score of baseline is 55.97008440927874 ± 2.3705057874918936


In [102]:
acc,prec,rec,f1,std_acc,std_prec,std_rec,std_f1 = cancer_modeling(ind_vars,dep_vars,model1)

model_type = "LR"

print(f"Accuracy of {model_type} model is",(acc*100),"±",(std_acc*100))
print(f"Precision of {model_type} is",(prec*100),"±",(std_prec*100))
print(f"Recall of {model_type} is",(rec*100),"±",(std_rec*100))
print(f"F1-score of {model_type} is",(f1*100),"±",(std_f1*100))

Accuracy of LR model is 85.40034932892075 ± 2.914169279206373
Precision of LR is 86.29602010072117 ± 2.8010791472801624
Recall of LR is 85.40034932892075 ± 2.914169279206373
F1-score of LR is 85.62503247414872 ± 2.864416357535586


In [103]:
acc,prec,rec,f1,std_acc,std_prec,std_rec,std_f1 = cancer_modeling(ind_vars,dep_vars,model2)

model_type = "SVC"

print(f"Accuracy of {model_type} model is",(acc*100),"±",(std_acc*100))
print(f"Precision of {model_type} is",(prec*100),"±",(std_prec*100))
print(f"Recall of {model_type} is",(rec*100),"±",(std_rec*100))
print(f"F1-score of {model_type} is",(f1*100),"±",(std_f1*100))

Accuracy of SVC model is 89.33443647729362 ± 2.66342949822163
Precision of SVC is 89.6499018926731 ± 2.7288133410971556
Recall of SVC is 89.33443647729362 ± 2.66342949822163
F1-score of SVC is 89.41673044338901 ± 2.6596191666235116


In [104]:
acc,prec,rec,f1,std_acc,std_prec,std_rec,std_f1 = cancer_modeling(ind_vars,dep_vars,model3)

model_type = "RF"

print(f"Accuracy of {model_type} model is",(acc*100),"±",(std_acc*100))
print(f"Precision of {model_type} is",(prec*100),"±",(std_prec*100))
print(f"Recall of {model_type} is",(rec*100),"±",(std_rec*100))
print(f"F1-score of {model_type} is",(f1*100),"±",(std_f1*100))

Accuracy of RF model is 90.21833057547343 ± 2.5644256817971995
Precision of RF is 90.76030393608934 ± 2.553024792848182
Recall of RF is 90.21833057547343 ± 2.5644256817971995
F1-score of RF is 90.345486091813 ± 2.534989944334591


In [105]:
acc,prec,rec,f1,std_acc,std_prec,std_rec,std_f1 = cancer_modeling(ind_vars,dep_vars,model4)

model_type = "GBC"

print(f"Accuracy of {model_type} model is",(acc*100),"±",(std_acc*100))
print(f"Precision of {model_type} is",(prec*100),"±",(std_prec*100))
print(f"Recall of {model_type} is",(rec*100),"±",(std_rec*100))
print(f"F1-score of {model_type} is",(f1*100),"±",(std_f1*100))

Accuracy of GBC model is 89.40292333149475 ± 2.7022381989473145
Precision of GBC is 89.95694756642433 ± 2.4313479931272055
Recall of GBC is 89.40292333149475 ± 2.7022381989473145
F1-score of GBC is 89.53730561629716 ± 2.6325561094439456
